# 03 — Pretrained Transformer Baseline: XLM-RoBERTa Plain (`noMLM+mean` & Optional `03b` `noMLM+cls`)

**Purpose:** Evaluate off-the-shelf `xlm-roberta-base` without domain-adaptive MLM pretraining (`noMLM+mean`, labeled **"XLM-R plain"** in the final results table), plus the optional stricter `[CLS]` pooling variant (`03b`: `noMLM+cls`).

- **Architecture & Hyperparameters (identical to the factorial ablation):**
  - Backbone: `xlm-roberta-base` (`MAX_LEN = 128`)
  - Pooling: Masked Mean Pooling (`noMLM+mean`) and optional `[CLS]` Pooling (`noMLM+cls`)
  - Classification Heads: Symmetric 2-layer GELU heads (`Linear(768, 384) -> GELU -> Dropout(0.3) -> Linear(384, K)`)
  - Loss: Inverse-frequency class-weighted Cross-Entropy (`W_SENT`, `W_INT`)
  - Optimization: Batch size `8` $\times$ Gradient Accumulation `2` (effective batch `16`), Encoder LR `2e-6`, Head LR `2e-5`, Weight Decay `0.01`, Dropout `0.3`, Max `10` epochs, Early Stopping Patience `3` on validation mean Macro-F1 $= (\text{F1}_{\text{sent}} + \text{F1}_{\text{int}}) / 2$.
  - Seeds: `42, 43, 44` across both `runA` and `runB`.
- **Cache Verification:** If cached `.npz` predictions from the ablation notebooks exist, verifies exact agreement with the published ablation tables:
  - **Run A (`noMLM+mean`):** Sent Acc `0.713 ± 0.005`, Intent Acc `0.454 ± 0.002`
  - **Run B (`noMLM+mean`):** Sent Acc `0.728 ± 0.007`, Intent Acc `0.469 ± 0.006`

## 1. Environment & Single Config Cell

In [ ]:
# !pip install -q transformers accelerate
# ── Colab / Repo-Root Setup ───────────────────────────────────────────────────
import os, sys, subprocess
if "google.colab" in sys.modules and not os.path.exists("eval_utils.py"):
    if not os.path.exists("/content/Sinhala-joint-sentiment-and-intent"):
        subprocess.run(["git", "clone", "https://github.com/ThiwankaChanditha/Sinhala-joint-sentiment-and-intent.git", "/content/Sinhala-joint-sentiment-and-intent"], check=True)
    os.chdir("/content/Sinhala-joint-sentiment-and-intent")
elif not os.path.exists("eval_utils.py"):
    for _cand in ["..", "../..", "/content/Sinhala-joint-sentiment-and-intent"]:
        if os.path.exists(os.path.join(_cand, "eval_utils.py")):
            os.chdir(_cand)
            break
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

import os
import gc
import math
import time
import json
import random
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

from eval_utils import (
    SENT_CLASSES,
    INTENT_CLASSES,
    NS,
    NI,
    SEEDS,
    log_environment,
    set_all_seeds,
    load_splits,
    macro_f1_present,
    evaluate_and_save,
    summarize_Across_seeds,
)

warnings.filterwarnings("ignore")

# ── Single Config Cell (Identical to Ablation Hyperparameters) ────────────────
MODEL_NAME     = "xlm-roberta-base"
MAX_LEN        = 128
FT_EPOCHS      = 10
FT_BATCH       = 8
FT_GRAD_ACC    = 2
FT_LR          = 2e-5
ENC_LR_MULT    = 0.1          # encoder LR = 2e-6, head LR = 2e-5
WEIGHT_DECAY   = 0.01
DROPOUT        = 0.3
NUM_ATTN_HEADS = 4
PATIENCE       = 3            # early stopping on val mean macro-F1 (sent, intent)

RUNS_TO_EXEC   = ["runA", "runB"]
RUN_CLS_03B    = False        # Set True to also run optional 03b [CLS] pooling variant (noMLM+cls)

WORK_DIR       = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
SILVER_CSV     = None
GOLD_CSV       = None
RESULTS_DIR    = os.path.join(WORK_DIR, "results")
PREDS_DIR      = os.path.join(WORK_DIR, "preds")
ABLATION_NPZ_DIRS = {
    "runA": os.path.join(WORK_DIR, "preds_runA"),
    "runB": os.path.join(WORK_DIR, "preds_runB"),
}
for d in (RESULTS_DIR, PREDS_DIR):
    os.makedirs(d, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
env_info = log_environment()
print("Using device:", DEVICE)

## 2. Shared Dataset, Pooler, and JointModel (Unchanged from Ablation Notebooks)

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

class DS(Dataset):
    def __init__(self, d):
        e = tok(d["text"].tolist(), truncation=True, max_length=MAX_LEN)
        self.ids, self.am = e["input_ids"], e["attention_mask"]
        self.s, self.i = d["sent_id"].tolist(), d["intent_id"].tolist()
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, k):
        return {"input_ids": self.ids[k], "attention_mask": self.am[k], "sent": self.s[k], "intent": self.i[k]}

def collate(batch):
    out = tok.pad([{"input_ids": b["input_ids"], "attention_mask": b["attention_mask"]} for b in batch], return_tensors="pt")
    out["sent"]   = torch.tensor([b["sent"] for b in batch])
    out["intent"] = torch.tensor([b["intent"] for b in batch])
    return out

class Pooler(nn.Module):
    def __init__(self, hidden, mode, heads, dropout):
        super().__init__()
        self.mode = mode
        if mode == "attn":
            self.attn = nn.MultiheadAttention(hidden, heads, dropout=dropout, batch_first=True)
            self.query_sent = nn.Parameter(torch.randn(1, 1, hidden))
            self.query_int  = nn.Parameter(torch.randn(1, 1, hidden))
        self.norm = nn.LayerNorm(hidden)

    def forward(self, H, mask):
        if self.mode == "attn":
            B, kpm = H.size(0), (mask == 0)
            outs = []
            for q in (self.query_sent, self.query_int):
                o, _ = self.attn(q.expand(B, -1, -1), H, H, key_padding_mask=kpm)
                outs.append(self.norm(o.squeeze(1)))
            return outs
        if self.mode == "cls":
            p = H[:, 0]
        else:
            m = mask.unsqueeze(-1).to(H.dtype)
            p = (H * m).sum(1) / m.sum(1).clamp(min=1)
        p = self.norm(p)
        return p, p

class JointModel(nn.Module):
    def __init__(self, enc_path, pooling):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(enc_path)
        h = self.encoder.config.hidden_size
        self.pool = Pooler(h, pooling, NUM_ATTN_HEADS, DROPOUT)
        fc = lambda: nn.Sequential(nn.Linear(h, h // 2), nn.GELU(), nn.Dropout(DROPOUT))
        self.sent_fc, self.int_fc = fc(), fc()
        self.sent_head, self.intent_head = nn.Linear(h // 2, NS), nn.Linear(h // 2, NI)

    def forward(self, input_ids, attention_mask):
        H = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        ps, pi = self.pool(H, attention_mask)
        return self.sent_head(self.sent_fc(ps)), self.intent_head(self.int_fc(pi))

@torch.no_grad()
def predict_probs(model, loader):
    model.eval()
    S, I = [], []
    for b in loader:
        s, i = model(b["input_ids"].to(DEVICE), b["attention_mask"].to(DEVICE))
        S.append(torch.softmax(s, -1).cpu())
        I.append(torch.softmax(i, -1).cpu())
    return torch.cat(S).numpy(), torch.cat(I).numpy()

def class_weights(ids, K):
    c = np.bincount(ids, minlength=K).astype(float)
    return torch.tensor(np.where(c > 0, c.sum() / (K * np.maximum(c, 1)), 0.0), dtype=torch.float)

## 3. Training & Cache Loader Function

In [ ]:
def run_or_load_plain_xlmr(run: str, pooling: str, seed: int, notebook_tag: str, model_tag: str):
    train_df, val_df, test_df = load_splits(run=run, silver_csv=SILVER_CSV, gold_csv=GOLD_CSV)
    ablation_variant_name = f"noMLM+{pooling}"

    npz_candidates = [
        os.path.join(ABLATION_NPZ_DIRS.get(run, "."), f"{ablation_variant_name}__seed{seed}.npz"),
        os.path.join(PREDS_DIR, f"{ablation_variant_name}_{run}_seed{seed}.npz"),
        os.path.join(PREDS_DIR, f"{ablation_variant_name}__seed{seed}.npz"),
    ]
    cached_npz = next((p for p in npz_candidates if os.path.exists(p)), None)

    if cached_npz is not None:
        print(f"[{run} | {ablation_variant_name} | seed {seed}] Loading cached predictions from {cached_npz}")
        arr = dict(np.load(cached_npz))
        ts, ti = arr["test_S"], arr["test_I"]
        best_ep = int(arr.get("best_epoch", -1))
        best_val = float(arr.get("best_val", -1.0))
    else:
        train_ds, val_ds, test_ds = DS(train_df), DS(val_df), DS(test_df)
        val_loader  = DataLoader(val_ds,  batch_size=FT_BATCH, shuffle=False, collate_fn=collate)
        test_loader = DataLoader(test_ds, batch_size=FT_BATCH, shuffle=False, collate_fn=collate)

        W_SENT = class_weights(train_df["sent_id"].values, NS).to(DEVICE)
        W_INT  = class_weights(train_df["intent_id"].values, NI).to(DEVICE)
        val_ys, val_yi = val_df["sent_id"].values, val_df["intent_id"].values

        set_all_seeds(seed)
        model = JointModel(MODEL_NAME, pooling).to(DEVICE)
        g = torch.Generator()
        g.manual_seed(seed)
        loader = DataLoader(train_ds, batch_size=FT_BATCH, shuffle=True, collate_fn=collate, generator=g)
        crit_s, crit_i = nn.CrossEntropyLoss(weight=W_SENT), nn.CrossEntropyLoss(weight=W_INT)
        head_params = [p for n, p in model.named_parameters() if not n.startswith("encoder.")]
        opt = torch.optim.AdamW(
            [{"params": model.encoder.parameters(), "lr": FT_LR * ENC_LR_MULT},
             {"params": head_params, "lr": FT_LR}],
            weight_decay=WEIGHT_DECAY,
        )
        total = FT_EPOCHS * math.ceil(len(loader) / FT_GRAD_ACC)
        sched = get_linear_schedule_with_warmup(opt, max(1, total // 10), total)

        best_val, best_ep, best_state, bad = -1.0, 0, None, 0
        t0 = time.time()
        for ep in range(FT_EPOCHS):
            model.train()
            tl = 0.0
            opt.zero_grad()
            for step, b in enumerate(loader):
                s, i = model(b["input_ids"].to(DEVICE), b["attention_mask"].to(DEVICE))
                loss = crit_s(s, b["sent"].to(DEVICE)) + crit_i(i, b["intent"].to(DEVICE))
                (loss / FT_GRAD_ACC).backward()
                tl += loss.item()
                if (step + 1) % FT_GRAD_ACC == 0 or step + 1 == len(loader):
                    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt.step()
                    sched.step()
                    opt.zero_grad()
            vs, vi = predict_probs(model, val_loader)
            f_s = macro_f1_present(val_ys, vs.argmax(1), NS)
            f_i = macro_f1_present(val_yi, vi.argmax(1), NI)
            score = (f_s + f_i) / 2.0
            print(f"  [{ablation_variant_name} {run} s{seed}] ep {ep+1:>2} | train {tl/len(loader):.4f} | val F1 sent {f_s:.3f} int {f_i:.3f} | mean {score:.3f}")
            if score > best_val:
                best_val, best_ep, bad = score, ep + 1, 0
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            else:
                bad += 1
                if bad >= PATIENCE:
                    print("  early stop")
                    break

        model.load_state_dict(best_state)
        vs, vi = predict_probs(model, val_loader)
        ts, ti = predict_probs(model, test_loader)
        save_npz_path = os.path.join(PREDS_DIR, f"{ablation_variant_name}_{run}_seed{seed}.npz")
        np.savez(save_npz_path, val_S=vs, val_I=vi, test_S=ts, test_I=ti, best_epoch=best_ep, best_val=best_val)
        print(f"done {ablation_variant_name} ({run}, seed={seed}): best ep {best_ep}, val {best_val:.3f}, {time.time()-t0:.0f}s")
        del model, opt, sched, best_state
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    payload = evaluate_and_save(
        notebook_name=notebook_tag,
        model_name=model_tag,
        run=run,
        seed=seed,
        test_df=test_df,
        pred_sent_id=ts.argmax(1),
        pred_intent_id=ti.argmax(1),
        sent_probs=ts,
        intent_probs=ti,
        extra_meta={"pooling": pooling, "use_mlm": False, "best_epoch": best_ep, "best_val_mean_f1": best_val},
        results_dir=RESULTS_DIR,
        preds_dir=PREDS_DIR,
    )
    return payload

## 4. Execute `noMLM+mean` (XLM-R Plain) Across Seeds 42, 43, 44 (Run A & Run B)

In [ ]:
all_summaries = []

for run in RUNS_TO_EXEC:
    seed_payloads = []
    for seed in SEEDS:
        p = run_or_load_plain_xlmr(
            run=run,
            pooling="mean",
            seed=seed,
            notebook_tag="03_xlmr_plain_baseline",
            model_tag="noMLM+mean",
        )
        seed_payloads.append(p)
    all_summaries.append(summarize_Across_seeds(seed_payloads))

    if RUN_CLS_03B:
        cls_payloads = []
        for seed in SEEDS:
            p_cls = run_or_load_plain_xlmr(
                run=run,
                pooling="cls",
                seed=seed,
                notebook_tag="03b_xlmr_cls_baseline",
                model_tag="noMLM+cls",
            )
            cls_payloads.append(p_cls)
        all_summaries.append(summarize_Across_seeds(cls_payloads))

display(pd.DataFrame(all_summaries)[["run", "model", "n_seeds", "sent_acc_str", "sent_f1_3cls_str", "sent_f1_4cls_str", "intent_acc_str", "intent_f1_str", "joint_acc_str"]])

## 5. Verification Against Published Ablation Tables (`noMLM+mean`)

In [ ]:
EXPECTED_ABLATION_REF = {
    "runA": {"sent_acc": 0.713, "sent_f1_4cls": 0.487, "sent_f1_3cls": 0.649, "intent_acc": 0.454, "intent_f1": 0.397, "joint_acc": 0.374},
    "runB": {"sent_acc": 0.728, "sent_f1_4cls": 0.494, "sent_f1_3cls": 0.659, "intent_acc": 0.469, "intent_f1": 0.407, "joint_acc": 0.390},
}

for s in all_summaries:
    if s["model"] != "noMLM+mean":
        continue
    run = s["run"]
    ref = EXPECTED_ABLATION_REF[run]
    print(f"=== Verification for {run} (noMLM+mean / XLM-R plain) ===")
    for k, exp_val in ref.items():
        got_val = round(s[f"{k}_mean"], 3)
        diff = abs(got_val - exp_val)
        status = "MATCH" if diff <= 0.002 else f"DIFF ({diff:.3f})"
        print(f"  {k:<15}: got={got_val:.3f} | expected={exp_val:.3f} -> {status}")